# Training Anomaly Detection Models

Interactive training for autoencoder, LSTM, and transformer anomaly detectors.
Models are trained on **normal data only** (semi-supervised approach).

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from torch.utils.data import DataLoader
from pathlib import Path

from src.data.dataset import NPPADWindowDataset
from src.models.anomaly import build_model
from src.training.trainer import Trainer
from src.utils.config import Config

## 1. Load Data

In [ ]:
processed_dir = Path('../data/processed')

train_dataset = NPPADWindowDataset(processed_dir / 'train.pt')
val_dataset = NPPADWindowDataset(processed_dir / 'val.pt')

# Filter to normal-only for training
train_normal = train_dataset.normal_only()

print(f'Training windows (normal only): {len(train_normal)}')
print(f'Validation windows (all): {len(val_dataset)}')
print(f'Window shape: {train_normal.windows.shape}')

## 2. Train a Model

Choose a config: `autoencoder.yaml`, `lstm.yaml`, or `transformer.yaml`

In [ ]:
# Select model
CONFIG_PATH = '../configs/autoencoder.yaml'  # Change this to try different models

config = Config.from_yaml(CONFIG_PATH)
model = build_model(config)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {config.model.name}')
print(f'Parameters: {n_params:,}')
print(model)

In [ ]:
train_loader = DataLoader(train_normal, batch_size=config.train.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.train.batch_size, shuffle=False)

trainer = Trainer(model, train_loader, val_loader, config)
metrics = trainer.train()
print(f'\nTraining complete: {metrics}')

## 3. Visualize Training Curves

Launch TensorBoard to view detailed curves:
```bash
tensorboard --logdir=../runs
```

In [ ]:
# Quick anomaly score check on a batch
import matplotlib.pyplot as plt

model.eval()
batch_windows, batch_labels = next(iter(val_loader))

device = next(model.parameters()).device
scores = model.anomaly_score(batch_windows.to(device)).cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 5))
normal_scores = scores[batch_labels == 0]
anomaly_scores = scores[batch_labels == 1]

if len(normal_scores) > 0:
    ax.hist(normal_scores, bins=30, alpha=0.6, label='Normal', color='green')
if len(anomaly_scores) > 0:
    ax.hist(anomaly_scores, bins=30, alpha=0.6, label='Anomaly', color='red')

ax.set_xlabel('Anomaly Score')
ax.set_ylabel('Count')
ax.set_title(f'{config.model.name} - Anomaly Score Distribution (sample batch)')
ax.legend()
plt.tight_layout()
plt.show()